# UXR Job Radar：在 Colab GPU 上批量处理积压岗位

本 notebook 用和本机完全相同的两阶段流程和校验代码，把 Google Drive 里导出的积压岗位交给官方的 **Qwen3.8-27B-FP8**（vLLM）判断，结果自动保存回 Google Drive。之后由本机导入，导入时会对每条结果重新校验一次。

**怎么运行**

1. 菜单 **代码执行程序 → 更改运行时类型**，选择 **G4 GPU**（RTX PRO 6000，96GB）。没有 G4 时可以选 A100（80GB 最好）。
2. **代码执行程序 → 全部运行**。第一次运行会请求访问 Google Drive，点同意即可。
3. 保持这个标签页开着。预计：安装和加载模型 10–20 分钟，处理积压 1–3 小时（以实际进度输出为准）。
4. 看到 `完成` 后告诉 Claude 导入。

**中途断线**：重新"全部运行"即可。程序会从 Drive 里的 `results-partial.sqlite3` 继续，已经完成的岗位不会重做。

**数据**：发送给模型的只有公开的岗位信息和匿名的筛选策略，不含任何个人资料。

In [ ]:
# Settings. The defaults match the batch exported to Google Drive.
DRIVE_DIR = "/content/drive/MyDrive/uxr-radar-colab"
MODEL = "Qwen/Qwen3.8-27B-FP8"   # official FP8 release of the weights the Mac runs as MLX 8-bit
CONCURRENCY = 48                 # parallel jobs; vLLM batches them on the GPU
CHUNK = 1500                     # jobs between checkpoints saved to Drive
BASE_URL = "http://127.0.0.1:8000/v1"

In [ ]:
import json, os, pathlib, subprocess
from google.colab import drive
drive.mount("/content/drive")
manifest = json.loads(pathlib.Path(DRIVE_DIR, "manifest.json").read_text())
print(json.dumps(manifest, indent=2))
assert MODEL in manifest["accepted_models"], "MODEL must be one of the manifest's accepted_models"
COMMIT = manifest["code_commit"]
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
# vLLM >= 0.17 serves Qwen3.8's hybrid attention. The project declares Python >= 3.13;
# its code is tested on 3.12, hence --ignore-requires-python.
!pip install -q -U vllm
!pip install -q httpx pydantic beautifulsoup4
!pip install -q --no-deps --ignore-requires-python "uxr-job-radar @ git+https://github.com/xthomaswang/uxr-job-radar@{COMMIT}"
import uxr_radar.core as core
assert core.PROMPT_VERSION == manifest["prompt_version"], "installed code does not match the exported batch"
print("prompt", core.PROMPT_VERSION)

In [ ]:
import time, httpx
log = open("/content/vllm.log", "w")
server = subprocess.Popen(["vllm", "serve", MODEL, "--host", "127.0.0.1", "--port", "8000",
    "--max-model-len", "16384", "--max-num-seqs", "64", "--gpu-memory-utilization", "0.92"],
    stdout=log, stderr=subprocess.STDOUT)
for _ in range(360):  # model download + load: up to 30 minutes
    try:
        if httpx.get("http://127.0.0.1:8000/health", timeout=5).status_code == 200:
            break
    except httpx.HTTPError:
        pass
    if server.poll() is not None:
        break
    time.sleep(5)
ready = server.poll() is None
print("vLLM ready" if ready else "vLLM failed to start; last log lines:")
if not ready:
    print("".join(open("/content/vllm.log").readlines()[-40:]))
    raise SystemExit("vLLM did not start")

In [ ]:
import contextlib, shutil, sqlite3, sys
from uxr_radar.core import Assessment, publication_decision
from uxr_radar.pipeline import pending_tasks, review_pending
from uxr_radar.store import Store

policy = json.loads(pathlib.Path(DRIVE_DIR, "policy.json").read_text())
work = pathlib.Path("/content/work"); work.mkdir(exist_ok=True)
db_path = work / "batch.sqlite3"
partial = pathlib.Path(DRIVE_DIR, "results-partial.sqlite3")
if not db_path.exists():  # resume from the Drive checkpoint after a disconnect
    shutil.copy(partial if partial.exists() else pathlib.Path(DRIVE_DIR, "batch.sqlite3"), db_path)
store = Store(db_path)

def checkpoint(target):
    snapshot = work / "snapshot.sqlite3"
    snapshot.unlink(missing_ok=True)
    copy = sqlite3.connect(snapshot); store.db.backup(copy); copy.close()
    shutil.copy(snapshot, str(target) + ".tmp"); os.replace(str(target) + ".tmp", target)

def counts():
    total = store.db.execute("SELECT count(*) FROM jobs").fetchone()[0]
    done = store.db.execute("SELECT count(*) FROM jobs WHERE assessment IS NOT NULL").fetchone()[0]
    return total, done

started = time.time(); start_done = counts()[1]
cell_out = sys.stdout  # per-job lines go to review.log; progress stays in the cell
with open("/content/review.log", "a") as review_log, contextlib.redirect_stdout(review_log):
    while pending_tasks(store, policy, MODEL):
        review_pending(store, policy, BASE_URL, MODEL, CHUNK, concurrency=CONCURRENCY)
        checkpoint(partial)
        total, done = counts()
        rate = (done - start_done) / max(1e-9, (time.time() - started) / 60)
        print(f"{done}/{total} assessed, {rate:.1f} jobs/min, ETA {(total - done) / max(rate, 1e-9):.0f} min", file=cell_out, flush=True)
    for _ in range(2):  # give failed stages two more chances before finishing
        review_pending(store, policy, BASE_URL, MODEL, 100000, retries_only=True, force=True, concurrency=CONCURRENCY)
    checkpoint(partial)
print("review finished", counts())

In [ ]:
import collections, vllm

def gpu_name():
    try:
        return subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    except OSError:
        return "unknown"

decisions = collections.Counter(publication_decision(Assessment.model_validate_json(a))
    for (a,) in store.db.execute("SELECT assessment FROM jobs WHERE assessment IS NOT NULL"))
total, done = counts()
summary = {"model": MODEL, "vllm": vllm.__version__, "concurrency": CONCURRENCY,
    "gpu": gpu_name(),
    "jobs": total, "assessed": done, "not_assessed": total - done, "decisions": dict(decisions),
    "minutes": round((time.time() - started) / 60, 1), "code_commit": COMMIT}
checkpoint(pathlib.Path(DRIVE_DIR, "results.sqlite3"))
pathlib.Path(DRIVE_DIR, "results-summary.json").write_text(json.dumps(summary, indent=2) + "\n")
server.terminate()
print(json.dumps(summary, indent=2))
print("完成：结果已保存到 Google Drive 的 uxr-radar-colab/results.sqlite3，可以告诉 Claude 导入了。")